# Обучение агента в Unity ML-Agents с экспортом ONNX

Этот ноутбук обучает агента из Unity-среды (`MyAgent?team=0`) с 6 наблюдениями и 2 непрерывными действиями (без прыжка). Используем PPO из `stable-baselines3` с поддержкой GPU и экспортируем модель в ONNX для Unity Sentis.

## Зависимости
- Установите библиотеки:
  ```bash
  pip install mlagents==0.30.0 stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html onnx numpy psutil
  ```
- Убедитесь, что путь к `UnityEnvironment.exe` правильный.
- Среда Unity должна быть собрана с `Behavior Name: MyAgent?team=0`, `Continuous Actions: 2`, `Discrete Actions: 0`.
- Python 3.8, NVIDIA GPU с CUDA 11.8 (или совместимая версия).
- Проверьте доступность GPU:
  ```bash
  python -c "import torch; print(torch.cuda.is_available())"
  ```

In [ ]:
# Запуск установки зависимостей внутри Jupyter-ноутбука через shell-команду (восклицательный знак).
# Эта команда установит mlagents (API Unity), stable-baselines3 (алгоритмы RL), gymnasium (интерфейс сред),
# и специфические бинарные колеса PyTorch с поддержкой CUDA 11.8 (суффикс +cu118). Также указывается индекс PyTorch wheel'ов через -f.
!pip3 install mlagents==0.30.0 stable-baselines3==2.0.0 gymnasium==0.28.1 torch==2.0.1+cu118 torchvision==0.15.2+cu118 -f https://download.pytorch.org/whl/torch_stable.html
# Примечание: выполнение этой ячейки изменит окружение Python и займет время. Убедитесь, что CUDA и драйверы соответствуют указанной версии PyTorch.

Looking in links: https://download.pytorch.org/whl/torch_stable.html


In [ ]:
# Импорт необходимых библиотек и модулей для работы с Unity, PyTorch и stable-baselines3.
import os  # Модуль для работы с путями и файловой системой
import numpy as np  # NumPy для массивов и численных операций
import torch  # PyTorch для тензоров и нейросетей
import torch.nn as nn  # Подмодуль нейронных слоёв PyTorch
from mlagents_envs.environment import UnityEnvironment  # Класс окружения Unity ML-Agents
from mlagents_envs.side_channel.engine_configuration_channel import EngineConfigurationChannel  # Канал для конфигурации движка Unity
from mlagents_envs.base_env import ActionTuple  # Контейнер для действий (continuous/discrete)
from stable_baselines3 import PPO  # Алгоритм PPO из stable-baselines3
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor  # Базовый класс для кастомных извлекателей признаков
from stable_baselines3.common.env_checker import check_env  # Утилита для проверки gym-совместимости среды
import gymnasium as gym  # Обёртка, совместимая с OpenAI Gym API
from gymnasium import spaces  # Конструкторы пространств (Box, Discrete и т.д.)

# Функция для корректного закрытия Unity-среды и аварийного завершения процессов при необходимости.
def close_unity_env(env):
    # Попытка корректно закрыть объект UnityEnvironment, если он существует.
    try:
        if env is not None:
            env.close()  # Вызываем метод close у UnityEnvironment для корректного завершения сессии
            print('Среда Unity успешно закрыта.')
        else:
            print('Среда не инициализирована.')
    except Exception as e:
        # Если при закрытии произошла ошибка, выводим её для отладки
        print(f'Ошибка при закрытии среды: {e}')
    finally:
        # В finally используем psutil, чтобы убедиться, что процессы Unity не остались запущены
        import psutil
        for proc in psutil.process_iter(['name']):
            # Если процесс точно называется UnityEnvironment.exe — безопасно принудительно завершить его
            if proc.info['name'] and proc.info['name'].lower() == 'unityenvironment.exe':
                proc.kill()
                print(f'Процесс UnityEnvironment.exe (PID: {proc.pid}) принудительно завершён.')
            # ВНИМАНИЕ: фильтрация по подстроке 'unity' может убить другие процессы Unity на машине; используйте осторожно
            if proc.info['name'] and 'unity' in proc.info['name'].lower():
                proc.kill()
                print(f"Процесс {proc.info['name']} (PID: {proc.pid}) принудительно завершён.")

# Формируем путь к исполняемому файлу Unity. Обратите внимание: строка ниже уже абсолютна, поэтому os.path.join с os.getcwd() не обязателен.
env_path = os.path.join(os.getcwd(), r'N:\\MyRL\\My_First_NPC\\Environment\\UnityEnvironment.exe')
# Альтернатива: env_path = r'N:\\MyRL\\My_First_NPC\\Environment\\UnityEnvironment.exe'

# Создаём канал конфигурации движка Unity, чтобы управлять скоростью симуляции и качеством для ускорения обучения
engine_channel = EngineConfigurationChannel()
# time_scale=20.0 ускоряет симуляцию в 20 раз, quality_level=0 ставит минимальный уровень качества для производительности
engine_channel.set_configuration_parameters(time_scale=20.0, quality_level=0)  # Уменьшил для стабильности

# Пытаемся инициализировать UnityEnvironment с указанными параметрами. Если что-то пойдёт не так — закроем процессы и пробросим исключение.
try:
    env = UnityEnvironment(file_name=env_path, worker_id=1, base_port=6000, side_channels=[engine_channel], timeout_wait=60)
    env.reset()  # Первый сброс среды после запуска
except Exception as e:
    # При ошибке инициализации выводим причину, пытаемся освободить ресурсы и повторно выбрасываем исключение
    print(f'Ошибка инициализации среды: {e}')
    close_unity_env(None)
    raise

# Получаем имя первого поведения (behavior) из спецификаций среды. Обычно в сборке ожидается одно поведение агента.
behavior_name = list(env.behavior_specs.keys())[0]
print(f'Behavior Name: {behavior_name}')  # Выводим имя поведения для отладки
spec = env.behavior_specs[behavior_name]  # Сохраняем спецификацию поведения для дальнейшего использования

# Печатаем размеры наблюдений и действий, чтобы убедиться, что они соответствуют ожиданиям (например: 6 наблюдений, 2 непрерывных действия)
print(f'Observation size: {spec.observation_specs[0].shape[0]}')
print(f'Continuous action size: {spec.action_spec.continuous_size}')
print(f'Discrete action branches: {spec.action_spec.discrete_branches}')

Behavior Name: MyAgent?team=0
Observation size: 6
Continuous action size: 2
Discrete action branches: ()


In [ ]:
# Кастомный извлекатель признаков (feature extractor) для stable-baselines3, наследуется от BaseFeaturesExtractor.
class CustomActorCriticNet(BaseFeaturesExtractor):
    # observation_space: объект gym.Space с описанием формы входного наблюдения
    # features_dim: размер выходного вектора признаков, который будет передан в политику/критик
    def __init__(self, observation_space, features_dim=128):
        # Вызываем конструктор базового класса с описанием observation_space и размером features_dim
        super(CustomActorCriticNet, self).__init__(observation_space, features_dim)
        # Создаём полносвязные слои. ВАЖНО: перемещение слоёв на device (GPU/CPU) осуществляется ниже при создании модели,
        # поэтому здесь мы НЕ обязаны вызывать .to(device) — если device ещё не определён, это приведёт к ошибке.
        # Если вы хотите явно перемещать слои сейчас, убедитесь, что переменная device определена выше.
        self.fc1 = nn.Linear(observation_space.shape[0], 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, features_dim)
        self.relu = nn.ReLU()  # Нелинейность ReLU между слоями

    # Прямой проход: применяем слои и активации в последовательности
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.fc3(x)  # Последний слой даёт features_dim-мерный вектор признаков
        return x

# policy_kwargs: аргументы для передачи в конструктор политики stable-baselines3. Здесь указываем наш extractor и архитектуру дополнительных слоёв
policy_kwargs = dict(
    features_extractor_class=CustomActorCriticNet,  # Использовать наш кастомный извлекатель признаков
    features_extractor_kwargs=dict(features_dim=128),  # Параметры конструктора извлекателя
    net_arch=[dict(pi=[64, 32], vf=[64, 32])]  # Дополнительные MLP слои для политики (pi) и value-функции (vf)
)

In [4]:
class UnityGymWrapper(gym.Env):
    def __init__(self, unity_env, behavior_name, spec):
        super(UnityGymWrapper, self).__init__()
        self.env = unity_env
        self.behavior_name = behavior_name
        self.spec = spec
        
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, shape=(spec.observation_specs[0].shape[0],), dtype=np.float32
        )
        self.action_space = spaces.Box(
            low=-1.0, high=1.0, shape=(spec.action_spec.continuous_size,), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.env.reset()
        decision_steps, _ = self.env.get_steps(self.behavior_name)
        obs = decision_steps.obs[0][0]
        info = {}
        return obs, info

    def step(self, action):
        # Ensure action is shaped (1, action_size)
        action = np.array(action, dtype=np.float32).reshape(1, -1)
        action_tuple = ActionTuple()
        action_tuple.add_continuous(action)  # Continuous actions only

        self.env.set_actions(self.behavior_name, action_tuple)
        self.env.step()

        decision_steps, terminal_steps = self.env.get_steps(self.behavior_name)
        done = len(terminal_steps) > 0
        if done:
            reward = float(terminal_steps.reward[0])
            obs = terminal_steps.obs[0][0]
        else:
            reward = float(decision_steps.reward[0])
            obs = decision_steps.obs[0][0]
        info = {}
        truncated = False

        return obs, reward, done, truncated, info

    def close(self):
        close_unity_env(self.env)

gym_env = UnityGymWrapper(env, behavior_name, spec)
check_env(gym_env)

In [ ]:
# Блок обучения: запускаем обучение модели PPO и гарантируем закрытие среды в любом случае
try:
    # Выбираем устройство: 'cuda' если PyTorch видит GPU, иначе 'cpu'
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Обучение на устройстве: {device}")

    # Создаём объект PPO и передаём ему среду и параметры обучения
    # 'MlpPolicy' — стандартная политика на основе MLP; policy_kwargs задаёт кастомный извлекатель и архитектуру
    model = PPO(
        'MlpPolicy',
        gym_env,
        policy_kwargs=policy_kwargs,  # наши кастомные параметры для политики
        learning_rate=3e-4,  # скорость обучения
        n_steps=2048,  # число шагов за эпизод, собираемых перед обновлением
        batch_size=64,  # размер батча для оптимизации
        n_epochs=10,  # количество эпох оптимизации на собранных данных
        gamma=0.99,  # фактор дисконтирования
        gae_lambda=0.95,  # параметр GAE
        clip_range=0.2,  # диапазон обрезки для PPO
        ent_coef=0.01,  # коэффициент энтропии (регуляризация политики)
        verbose=1,  # логирование
        device=device  # устройство вычислений (cpu или cuda)
    )

    # Запускаем процесс обучения на указанное количество шагов (total_timesteps)
    model.learn(total_timesteps=100000)
    # Сохраняем модель на диск (формат SB3)
    model.save('ppo_myagent_gpu')

    # Если используется GPU, выводим дополнительную информацию для диагностики
    if device == 'cuda':
        print(f"Используется GPU: {torch.cuda.get_device_name(0)}")
        print(f"Память GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

except Exception as e:
    # Ловим и печатаем любые ошибки обучения (например: проблемы с подключением к Unity или с GPU)
    print(f'Ошибка обучения: {e}')
finally:
    # В блоке finally обязательно закрываем Unity-среду, чтобы не оставлять процессы
    close_unity_env(env)

Обучение на устройстве: cuda
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


c:\Users\User\.conda\envs\ML_Agents\lib\site-packages\stable_baselines3\common\policies.py:460: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 18.9     |
|    ep_rew_mean     | 0.278    |
| time/              |          |
|    fps             | 177      |
|    iterations      | 1        |
|    time_elapsed    | 11       |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 18.1        |
|    ep_rew_mean          | 0.396       |
| time/                   |             |
|    fps                  | 150         |
|    iterations           | 2           |
|    time_elapsed         | 27          |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008468723 |
|    clip_fraction        | 0.0565      |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.83       |
|    explained_variance   | -0.256      |
|    learning_rate        | 0.

KeyboardInterrupt: 

In [ ]:
import torch  # Уже импортирован ранее, повторный импорт безопасен
from torch.nn import Parameter  # Parameter позволяет включать константы в граф ONNX

# WrapperNet — обёртка вокруг policy, чтобы привести выходы к формату, ожидаемому Unity/Sentis
class WrapperNet(torch.nn.Module):
    def __init__(self, policy, continuous_action_size):
        super(WrapperNet, self).__init__()
        # policy: объект политики из stable-baselines3 (может потребоваться адаптация интерфейса)
        self.policy = policy

        # version_number: версия формата ML-Agents/ONNX (пример: 3). Сохраняем как параметр, чтобы он был включён в экспортируемый граф
        version_number = torch.tensor([3], dtype=torch.float32).to(device)
        self.version_number = Parameter(version_number, requires_grad=False)

        # memory_size: размер внутренней памяти (RNN). У нас её нет -> 0
        memory_size = torch.tensor([0], dtype=torch.float32).to(device)
        self.memory_size = Parameter(memory_size, requires_grad=False)

        # continuous_action_output_shape: указываем форму выхода непрерывных действий (например [2])
        continuous_shape = torch.tensor([continuous_action_size], dtype=torch.float32).to(device)
        self.continuous_shape = Parameter(continuous_shape, requires_grad=False)

    def forward(self, obs, mask):
        # Вызов политики: в разных версиях SB3 интерфейс может отличаться; обычно policy(obs) возвращает (actions, ...)
        continuous_actions = self.policy(obs, deterministic=True)[0]
        # Применяем маску к действиям; это позволяет action_masks участвовать в графе ONNX
        continuous_actions = torch.mul(continuous_actions, mask)  # Фиктивное умножение
        # Возвращаем набор выходов в порядке, ожидаемом Unity
        return continuous_actions, self.continuous_shape, self.version_number, self.memory_size

# Экспорт в ONNX: оборачиваем политику и трассируем граф с помощью dummy-входов
try:
    policy = model.policy.to(device)  # Перемещаем политику на нужное устройство
    continuous_action_size = spec.action_spec.continuous_size  # Размер непрерывного действия (например, 2)
    wrapper_net = WrapperNet(policy, continuous_action_size)
    
    # Dummy inputs: батч из одного наблюдения и маска действий (все единицы)
    dummy_input = torch.randn(1, spec.observation_specs[0].shape[0]).to(device)  # [1, obs_dim]
    dummy_mask = torch.ones(1, continuous_action_size).to(device)  # [1, action_dim]
    
    # Экспортируем в ONNX. Имена входов/выходов и dynamic_axes важны для корректного импорта в Unity
    torch.onnx.export(
        wrapper_net,
        (dummy_input, dummy_mask),
        'trained_myagent.onnx',
        input_names=['obs_0', 'action_masks'],
        output_names=['continuous_actions', 'continuous_action_output_shape', 'version_number', 'memory_size'],
        dynamic_axes={
            'obs_0': {0: 'batch'},
            'action_masks': {0: 'batch'},
            'continuous_actions': {0: 'batch'},
            'continuous_action_output_shape': {0: 'batch'},
            'version_number': {0: 'batch'},
            'memory_size': {0: 'batch'}
        },
        opset_version=9,  # Если импорт падает — попробуйте 11 или 12
        verbose=False
    )
    print('Модель успешно сохранена: trained_myagent.onnx')
    print('Файл существует:', os.path.exists('trained_myagent.onnx'))
except Exception as e:
    print(f'Ошибка экспорта ONNX: {e}')
    print('Попробуйте opset_version=11 или проверьте версию Unity Sentis.')
finally:
    close_unity_env(env)

In [ ]:
# Прогон обученной модели по среде для тестирования — детерминированное выполнение действий
try:
    # Получаем начальное наблюдение через reset()
    obs, _ = gym_env.reset()
    for _ in range(1000):  # Лимит шагов теста — можно увеличить или заменить условием по эпизодам
        # Получаем действие от модели. deterministic=True отключает случайность в политике
        action, _ = model.predict(obs, deterministic=True)
        # Выполняем шаг в обёрнутой среде и получаем новые значения
        obs, reward, done, truncated, info = gym_env.step(action)
        # Если эпизод завершился — печатаем и перезапускаем среду
        if done or truncated:
            print('Эпизод завершён')
            obs, _ = gym_env.reset()
except Exception as e:
    # Отлавливаем ошибки, например проблемы с форматом входов/выходов или падение среды
    print(f'Ошибка тестирования: {e}')
finally:
    # Всегда закрываем Unity-среду после теста
    close_unity_env(env)

Ошибка тестирования: No Unity environment is loaded.
Ошибка при закрытии среды: No Unity environment is loaded.
